In [ ]:
import os
import random
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool, GCNConv
from sklearn.metrics import r2_score

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def safe_torch_load(path):
    try:
        return torch.load(path, weights_only=False)
    except TypeError:
        return torch.load(path)

def atom_feature_masking(x, batch_index, mask_ratio=0.10):
    """Zero all node features for approximately 10% of atoms per graph.

    At least one atom remains unmasked; edges are unchanged.
    A fresh mask is sampled for each training forward pass.
    """

    if mask_ratio <= 0:
        return x

    x_masked = x.clone()

    graph_ids = torch.unique(batch_index)

    for graph_id in graph_ids:

        node_idx = torch.where(
            batch_index == graph_id
        )[0]

        num_nodes = node_idx.numel()

        if num_nodes <= 1:
            continue


        num_mask = int(round(num_nodes * mask_ratio))


        num_mask = max(1, num_mask)


        num_mask = min(
            num_mask,
            num_nodes - 1
        )


        perm = torch.randperm(
            num_nodes,
            device=x.device
        )

        mask_nodes = node_idx[
            perm[:num_mask]
        ]


        x_masked[mask_nodes] = 0.0

    return x_masked


class SimpleGNN(nn.Module):

    def __init__(
        self,
        node_dim,
        edge_dim,
        global_dim,
        hidden_dims,
        dropout=0.2
    ):
        super().__init__()

        self.node_norm = nn.BatchNorm1d(node_dim)

        if edge_dim:
            self.edge_norm = nn.BatchNorm1d(edge_dim)

        if global_dim:
            self.global_norm = nn.BatchNorm1d(global_dim)

            self.global_mlp = nn.Sequential(
                nn.Linear(
                    global_dim,
                    hidden_dims[-1]
                ),
                nn.ReLU(),
                nn.Dropout(dropout)
            )

        self.convs = nn.ModuleList()

        in_dim = node_dim

        for h_dim in hidden_dims:
            self.convs.append(
                GCNConv(
                    in_dim,
                    h_dim
                )
            )
            in_dim = h_dim

        self.dropout = nn.Dropout(dropout)

        self.final_dim = (
            hidden_dims[-1] * 2
            if global_dim
            else hidden_dims[-1]
        )

        self.output_mlp = nn.Sequential(
            nn.Linear(
                self.final_dim,
                self.final_dim // 2
            ),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(
                self.final_dim // 2,
                1
            )
        )


    def forward(
        self,
        data,
        return_feat=False,
        atom_mask_ratio=0.0
    ):


        x = self.node_norm(data.x)


        if (
            self.training
            and atom_mask_ratio > 0
        ):
            x = atom_feature_masking(
                x=x,
                batch_index=data.batch,
                mask_ratio=atom_mask_ratio
            )


        if (
            hasattr(data, "edge_attr")
            and data.edge_attr is not None
        ):
            _ = self.edge_norm(
                data.edge_attr
            )


        u = (
            data.u
            if hasattr(data, "u")
            else None
        )

        if u is not None:
            u = self.global_norm(u)


        for conv in self.convs:

            x = F.relu(
                conv(
                    x,
                    data.edge_index
                )
            )

            x = self.dropout(x)


        node_pool = global_mean_pool(
            x,
            data.batch
        )


        if u is not None:
            h = torch.cat(
                [
                    node_pool,
                    self.global_mlp(u)
                ],
                dim=1
            )
        else:
            h = node_pool


        out = self.output_mlp(h).squeeze(-1)

        if return_feat:
            return out, h

        return out


def create_data_loader(
    graph_data,
    batch_size=32,
    shuffle=True
):

    data_list = []

    for graph in graph_data:

        data_list.append(
            Data(
                x=graph["x"],
                edge_index=graph["edge_index"],
                edge_attr=graph.get(
                    "edge_attr",
                    None
                ),
                u=graph.get(
                    "u",
                    None
                ),
                y=graph["y"]
            )
        )

    return DataLoader(
        data_list,
        batch_size=batch_size,
        shuffle=shuffle
    )


def train_model(
    train_data_dir,
    val_data_dir,
    save_path,
    epochs=1000,
    batch_size=32,
    lr=1e-4,
    min_lr=1e-6,
    hidden_dims=[64, 64],
    dropout=0.2,
    lr_patience=10,
    es_patience=100,
    atom_mask_ratio=0.10
):

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    print(
        f"Device: {device}"
    )

    print(
        f"Atom feature masking ratio: "
        f"{atom_mask_ratio:.0%}"
    )


    train_graph_data = safe_torch_load(
        os.path.join(
            train_data_dir,
            "graph_data.pt"
        )
    )

    val_graph_data = safe_torch_load(
        os.path.join(
            val_data_dir,
            "graph_data.pt"
        )
    )


    train_y = torch.stack(
        [
            g["y"]
            for g in train_graph_data
        ]
    ).view(-1)

    y_mean = train_y.mean().item()

    y_std = (
        train_y.std().item()
        + 1e-8
    )


    for g in train_graph_data:

        g["y"] = (
            g["y"]
            - y_mean
        ) / y_std


    for g in val_graph_data:

        g["y"] = (
            g["y"]
            - y_mean
        ) / y_std


    train_loader = create_data_loader(
        train_graph_data,
        batch_size=batch_size,
        shuffle=True
    )


    val_loader = create_data_loader(
        val_graph_data,
        batch_size=batch_size,
        shuffle=False
    )


    sample = train_graph_data[0]

    node_dim = sample["x"].size(1)

    edge_attr = sample.get(
        "edge_attr",
        None
    )

    edge_dim = (
        edge_attr.size(1)
        if edge_attr is not None
        else 0
    )

    u_sample = sample.get(
        "u",
        None
    )

    global_dim = (
        u_sample.size(1)
        if u_sample is not None
        else 0
    )


    print(
        f"node_dim   = {node_dim}"
    )

    print(
        f"edge_dim   = {edge_dim}"
    )

    print(
        f"global_dim = {global_dim}"
    )


    model = SimpleGNN(
        node_dim=node_dim,
        edge_dim=edge_dim,
        global_dim=global_dim,
        hidden_dims=hidden_dims,
        dropout=dropout
    ).to(device)


    optimizer = optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=1e-5
    )


    scheduler = (
        optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=lr_patience,
            min_lr=min_lr
        )
    )


    best_r2 = -float("inf")

    no_improve = 0


    for epoch in range(
        1,
        epochs + 1
    ):


        model.train()

        epoch_loss = 0.0


        for batch in train_loader:

            batch = batch.to(device)

            optimizer.zero_grad()


            pred, _ = model(
                batch,
                return_feat=True,
                atom_mask_ratio=atom_mask_ratio
            )


            loss = F.mse_loss(
                pred,
                batch.y.view(-1)
            )


            loss.backward()

            optimizer.step()


            epoch_loss += loss.item()


        avg_loss = (
            epoch_loss
            / len(train_loader)
        )


        model.eval()

        ys_val = []

        preds_val = []


        with torch.no_grad():

            for batch in val_loader:

                batch = batch.to(device)

                out = model(batch)

                ys_val.append(
                    batch.y
                    .view(-1)
                    .cpu()
                    .numpy()
                )

                preds_val.append(
                    out
                    .view(-1)
                    .cpu()
                    .numpy()
                )


        val_r2 = r2_score(
            np.concatenate(
                ys_val
            ),
            np.concatenate(
                preds_val
            )
        )


        current_lr = (
            optimizer
            .param_groups[0]["lr"]
        )


        print(
            f"Epoch {epoch:4d} | "
            f"Loss: {avg_loss:.4f} | "
            f"Val R²: {val_r2:.4f} | "
            f"LR: {current_lr:.2e}"
        )


        scheduler.step(
            epoch_loss
        )


        if val_r2 > best_r2:

            best_r2 = val_r2

            no_improve = 0


            torch.save(
                {
                    "model_state_dict":
                        model.state_dict(),

                    "y_mean":
                        y_mean,

                    "y_std":
                        y_std,

                    "node_dim":
                        node_dim,

                    "edge_dim":
                        edge_dim,

                    "global_dim":
                        global_dim,

                    "hidden_dims":
                        hidden_dims,

                    "dropout":
                        dropout,

                    "atom_mask_ratio":
                        atom_mask_ratio,

                    "best_val_r2":
                        best_r2
                },
                save_path
            )


            print(
                f"Saved best model; "
                f"Val R² = "
                f"{best_r2:.4f}"
            )


        else:

            no_improve += 1


            if (
                no_improve
                >= es_patience
            ):

                print(
                    f"Early stopping: validation R² "
                    f"for {es_patience} "
                    f"epochs without improvement."
                )

                break


    print(
        f"\nTraining complete"
    )

    print(
        f"Best Val R² = "
        f"{best_r2:.4f}"
    )

    print(
        f"Model saved at:"
    )

    print(
        save_path
    )


    return (
        best_r2,
        save_path
    )


if __name__ == '__main__':
    seeds = [0, 8, 42, 100, 456, 618, 1189, 2025, 2077, 2048]
    project_root = os.path.abspath(os.getcwd())
    train_data_dir = os.path.join(project_root, 'data-set', 'train')
    val_data_dir = os.path.join(project_root, 'data-set', 'validation')
    save_dir = os.path.join(project_root, 'results', 'atom_masking')
    os.makedirs(save_dir, exist_ok=True)
    atom_mask_ratio = 0.10

    fixed_config = {
        'epochs': 5000, 'batch_size': 64, 'lr': 1e-3,
        'min_lr': 1e-4, 'hidden_dims': [128, 128], 'dropout': 0.1,
        'lr_patience': 20, 'es_patience': 100,
        'atom_mask_ratio': atom_mask_ratio,
    }
    manifest_path = os.path.join(save_dir, 'checkpoint_manifest.csv')
    with open(manifest_path, 'w', newline='', encoding='utf-8') as manifest_file:
        writer = csv.DictWriter(manifest_file, fieldnames=['seed', 'checkpoint_path'])
        writer.writeheader()
        for seed in seeds:
            set_seed(seed)
            save_path = os.path.join(save_dir, f'gnn_atommask10_seed({seed})_128_128_0.1.pt')
            print(f'Training atom-masked GCN: seed {seed}.')
            train_model(
                train_data_dir=train_data_dir, val_data_dir=val_data_dir,
                save_path=save_path, **fixed_config,
            )
            writer.writerow({'seed': seed, 'checkpoint_path': os.path.relpath(save_path, project_root)})
            manifest_file.flush()
    print(f'Atom-masking experiment completed. Checkpoint manifest: {manifest_path}')
